### Data Reading from CSV (Volumes)

In [0]:
df = spark.read.format("csv").option('inferSchema', True)\
        .option('header', True)\
        .load('/Volumes/deltalake/default/raw/BigMart Sales.csv')

df.display()


## Data Reading JSON

In [0]:
df_json = spark.read.format("json").option('inferSchema', True)\
                    .option('header', True)\
                    .option('multiline', False)\
                    .load('/Volumes/deltalake/default/raw/drivers.json')

df_json.display()


## SCHEMA - DDL 
## Method 1 : DDL Schema

In [0]:
df.printSchema()

In [0]:
my_ddl_schema = '''
        Item_Identifier string,
        Item_Weight string,
        Item_Fat_Content string,
        Item_Visibility double,
        Item_Type string,
        Item_MRP double,
        Outlet_Identifier string,
        Outlet_Establishment_Year integer,
        Outlet_Size string,
        Outlet_Location_Type string,
        Outlet_Type string,
        Item_Outlet_Sales double
'''
df = spark.read.format('csv')\
                .schema(my_ddl_schema)\
                .option('header', True)\
                .load('/Volumes/deltalake/default/raw/BigMart Sales.csv')
df.display()

### Method 2 : StructType() Schema

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
my_struct_schema = StructType([
    StructField('Item_identifier', StringType(), True),
    StructField('Item_Weight', StringType(), True),
    StructField('Item_Fat_Content', StringType(), True),
    StructField('Item_Visibility', StringType(), True),
    StructField('Item_Type', StringType(), True),
    StructField('Item_MRP', StringType(), True),
    StructField('Outlet_Identifier', StringType(), True),
    StructField('Outlet_Establishment_Year', StringType(), True),
    StructField('Outlet_Size', StringType(), True),
    StructField('Outlet_Location_Type', StringType(), True),
    StructField('Outlet_Type', StringType(), True),
    StructField('Item_Outlet_Sales', StringType(), True)
])

df = spark.read.format('csv')\
                .schema(my_struct_schema)\
                .option('header', True)\
                .load('/Volumes/deltalake/default/raw/BigMart Sales.csv')
                
df.display()


## **SELECT**
## Method 1

In [0]:
df_sel = df.select('Item_Identifier', 'Item_Weight', 'Item_Fat_Content').display()

## **Method 2** : using col 


In [0]:
df.select(col('Item_Identifier'), col('Item_Weight'), col('Item_Fat_Content')).display()

## Method 3 : SelectExpr

In [0]:
df.selectExpr('Item_Identifier', 'Item_Weight', 'Item_Fat_Content').display()

## Alias 

In [0]:
df.select(col('Item_Identifier').alias('Item_ID'), 
          col('Item_Weight').alias('Item_Wt'), 
          col('Item_Fat_Content').alias('Fat_Content')).display()

In [0]:
# where/filter transformation is used to filter rows based on a condition
# Both where() and filter() are equivalent in PySpark

# Example: Filter rows where Item_Weight > 10
df_filtered = df.where(col('Item_Weight') > 10)
display(df_filtered)

# Alternatively, using filter()
df_filtered2 = df.filter(col('Item_Weight') > 10)
# display(df_filtered2)

# Example: Apply two filters - Item_Weight > 10 and Item_Fat_Content == 'Low Fat'
df_filtered_multi = df.where((col('Item_Weight') > 10) & (col('Item_Fat_Content') == 'Low Fat'))
display(df_filtered_multi)

# Alternatively, using filter() with two conditions
df_filtered_multi2 = df.filter((col('Item_Weight') > 10) & (col('Item_Fat_Content') == 'Low Fat'))
# display(df_filtered_multi2)

# Senario 3 : Outlet_Location_Type = Tier1 or Tier 2 and OutletSize = null records
df_filtered3 = df.where(col('Outlet_Location_Type').isin(['Tier 1', 'Tier 2']) & col('Outlet_Size').isNull())
display(df_filtered3)



## **Rename Columns** : withColumnRenamed()

In [0]:
df.withColumnRenamed('Item_Weight', 'ItemWt').display()
# Note it changes column name at Dataframe level

## **_Create a new column_** : withColumn

In [0]:
## Senario 1 : Create a new constant column
# To add any constant value we use lit() function
df.withColumn('Flag', lit('new')).display()

In [0]:
# Senario 2
# Create a new column with multiple of Item_Wight and Item_MRP
df.withColumn('multiply', col('Item_Weight')*col('Item_MRP')).display()

In [0]:
# Senario 2 : Replace low fat with LF and Regular with Reg
df.withColumn('Item_Fat_Content', regexp_replace(col('Item_Fat_Content'), 'Regular', 'Reg'))\
    .withColumn('Item_Fat_Content', regexp_replace(col('Item_Fat_Content'), 'Low Fat', 'LF'))\
    .withColumn('Item_Fat_Content', regexp_replace(col('Item_Fat_Content'), 'low fat', 'LF')).display()


## Type Casting 

In [0]:
df_new = df.withColumn('Item_Weight', col('Item_Weight').cast(IntegerType()))
df_new.display()

## Sort/Order By : .sort(col(<col_name>).desc()), .sort(col(<col_name>).desc())

In [0]:
df.sort(col('Item_weight').desc()).display()

In [0]:
df.sort(col('Item_Visibility').asc()).display()

In [0]:
# Sorting based on multiple columns and both of them are in descending order
df.sort(['Item_Weight', 'Item_Visibility'], ascending=[0, 0]).display()

In [0]:
# Sorting based on multiple columns and both of them are in descending order
df.sort(['Item_Weight', 'Item_Visibility'], ascending=[0, 1]).display()

## Limit

In [0]:
df.limit(10).display()

## DROP
### Senario 1 : Drop 1 column

In [0]:
df.drop('Item_Visibility').display()

In [0]:
# Scenario 2 : Drop multiple columns

df.drop(col('Item_Visibility'), col('Item_Type'), col('Item_MRP')).display()

## Drop Duplicates

In [0]:
# Drop duplicate rows from the dataframe
# This process is also called Dedup
# distinct() is also having the same functionality
df.dropDuplicates().display()

In [0]:
df.dropDuplicates(['Item_Identifier', 'Item_Type']).display()

In [0]:
df.drop_duplicates(subset=['Item_Type']).display()